# Content Based Filtering



Recommendation systems are a collection of algorithms used to recommend items to users based on information taken from the user. These systems have become ubiquitous, and can be commonly seen in online stores, movies databases and job finders. In this notebook, we will explore Content-based recommendation systems and implement a simple version of one using Python and the Pandas library.


### Table of contents

<div class="alert alert-block alert-info" style="margin-top: 20px">
    <ol>
        <li><a href="https://#ref1">Acquiring the Data</a></li>
        <li><a href="https://#ref2">Preprocessing</a></li>
        <li><a href="https://#ref3">Content-Based Filtering</a></li>
    </ol>
</div>
<br>


<a id="ref1"></a>

# Acquiring the Data


In [2]:
import pandas as pd
import os

In [3]:
#Storing the movie information into a pandas dataframe
# movies_df = pd.read_csv('movies.csv')
#Storing the user information into a pandas dataframe
# ratings_df = pd.read_csv('ratings.csv')

# movies_df.head()

In [17]:
dataset_path = os.path.join(os.path.expanduser('~'), 'datasets', 'movies', '7')

credits_df = pd.read_csv(dataset_path + '/credits.csv')
# movies_meta_df = pd.read_csv(dataset_path + '/movies_metadata.csv')
# ratings_df = pd.read_csv(dataset_path + '/ratings.csv')

In [217]:
dataset_path_2 = os.path.join(os.path.expanduser('~'), 'datasets', 'ml-32m')
ratings_df = pd.read_csv(dataset_path_2 + '/ratings.csv')

 
# links = pd.read_csv(dataset_path_2 + '/links.csv')
# 
# tags = pd.read_csv(dataset_path_2 + '/tags.csv')
# tags.head(40)

In [218]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [12]:
links

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0
...,...,...,...
87580,292731,26812510,1032473.0
87581,292737,14907358,986674.0
87582,292753,12388280,948139.0
87583,292755,64027,182776.0


In [15]:
ratings_2 = pd.read_csv(dataset_path_2 + '/ratings.csv')
ratings_2.head(10)

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858
5,1,34,2.0,943228491
6,1,36,1.0,944249008
7,1,80,5.0,944248943
8,1,110,3.0,943231119
9,1,111,5.0,944249008


In [16]:
ratings_2.shape

(32000204, 4)

In [6]:
credits_df

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862
...,...,...,...
45471,"[{'cast_id': 0, 'character': '', 'credit_id': ...","[{'credit_id': '5894a97d925141426c00818c', 'de...",439050
45472,"[{'cast_id': 1002, 'character': 'Sister Angela...","[{'credit_id': '52fe4af1c3a36847f81e9b15', 'de...",111109
45473,"[{'cast_id': 6, 'character': 'Emily Shaw', 'cr...","[{'credit_id': '52fe4776c3a368484e0c8387', 'de...",67758
45474,"[{'cast_id': 2, 'character': '', 'credit_id': ...","[{'credit_id': '533bccebc3a36844cf0011a7', 'de...",227506


In [7]:
metadata_df = pd.read_csv(f"{dataset_path}/movies_metadata.csv", nrows=10)#%%
metadata_df

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173
5,False,NaN,60000000,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...",NaN,949,tt0113277,en,Heat,"Obsessive master thief, Neil McCauley leads a ...",...,1995-12-15,187436818,170.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,A Los Angeles Crime Saga,Heat,False,7.7,1886
6,False,NaN,58000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...",NaN,11860,tt0114319,en,Sabrina,An ugly duckling having undergone a remarkable...,...,1995-12-15,0,127.0,"[{'iso_639_1': 'fr', 'name': 'Français'}, {'is...",Released,You are cordially invited to the most surprisi...,Sabrina,False,6.2,141
7,False,NaN,0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",NaN,45325,tt0112302,en,Tom and Huck,"A mischievous young boy, Tom Sawyer, witnesses...",...,1995-12-22,0,97.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,The Original Bad Boys.,Tom and Huck,False,5.4,45
8,False,NaN,35000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",NaN,9091,tt0114576,en,Sudden Death,International action superstar Jean Claude Van...,...,1995-12-22,64350171,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Terror goes into overtime.,Sudden Death,False,5.5,174
9,False,"{'id': 645, 'name': 'James Bond Collection', '...",58000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.mgm.com/view/movie/757/Goldeneye/,710,tt0113189,en,GoldenEye,James Bond must unmask the mysterious head of ...,...,1995-11-16,352194034,130.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,No limits. No fears. No substitutes.,GoldenEye,False,6.6,1194


In [19]:
# Dropping useless bits from movies_meta_df
movies_df = pd.read_csv(f"{dataset_path}/movies_metadata.csv", usecols=['id', 'title', 'release_date', 'genres', 'popularity', 'vote_average', 'vote_count'])

/var/folders/lr/44zcpqfx2196g_qy5t3kc7m40000gp/T/ipykernel_28406/1038718327.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_df = pd.read_csv(f"{dataset_path}/movies_metadata.csv", usecols=['id', 'title', 'release_date', 'genres', 'popularity', 'vote_average', 'vote_count'])


In [20]:
movies_df.shape

(45466, 7)

In [21]:
# from google.colab import drive
# drive.mount('/content/drive')

In [22]:
# ratings_small_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/7/ratings_small.csv')

In [23]:
# movies_meta_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/7/movies_metadata.csv')
# movies_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/movies/movies.csv')

In [24]:
# ratings_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/movies/ratings.csv')

In [25]:
# credits_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/7/credits.csv')
# credits_df.head()

In [26]:
movies_df = movies_df.sort_values(by='release_date', ascending=False)
# movies_meta_df.head(3)
movies_df = movies_df.drop(35587) # A weird film entry is now gone!
movies_df.head(3)

,genres,id,popularity,release_date,title,vote_average,vote_count
26559,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",76600,6.020055,2020-12-16,Avatar 2,0.0,58.0
38885,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",299782,0.238154,2018-12-31,The Other Side of the Wind,0.0,1.0
30402,"[{'id': 53, 'name': 'Thriller'}, {'id': 28, 'n...",38700,2.178546,2018-11-07,Bad Boys for Life,0.0,12.0


<a id="ref2"></a>

# Preprocessing


In [27]:

from math import sqrt
import numpy as np
import matplotlib.pyplot as plt

import ast
%matplotlib inline

First, let's get all of the imports out of the way:


Now let's read each file into their Dataframes:


In [28]:
# Make a genres table / format it to be able to use it as a feature


In [219]:
# prompt: print the dimensions of ratings, credits, movies, movies_metadata

print("ratings_df dimensions:", ratings_df.shape)
print("credits_df dimensions:", credits_df.shape)
print("movies_df dimensions:", movies_df.shape)
# print("movies_meta_df dimensions:", movies_meta_df.shape)


ratings_df dimensions: (32000204, 4)
credits_df dimensions: (45476, 3)
movies_df dimensions: (45465, 7)


In [30]:


def get_first_3_cast(cast_series):
    """
    Extracts names and IDs of the first 3 cast members from a Pandas Series.

    Args:
        cast_series (pandas.Series): A Pandas Series containing cast information
                                      (string representation of list of dictionaries).

    Returns:
        pandas.DataFrame: DataFrame with a column 'cast_info' containing tuples of (name, ID)
                         for the first 3 cast members.
    """

    all_cast_info = []

    for cast_list_str in cast_series:
        cast_list = ast.literal_eval(cast_list_str)  # Convert string to list
        cast_info = []
        for i in range(min(3, len(cast_list))):
            cast_info.append((cast_list[i]['name'], cast_list[i]['id']))  # Create (name, ID) tuple
        all_cast_info.append(cast_info)

    return pd.DataFrame({'cast_info': all_cast_info})


# Usage:
cast_info_df = get_first_3_cast(credits_df['cast'])
cast_info_df.head()

,cast_info
0,"[(Tom Hanks, 31), (Tim Allen, 12898), (Don Ric..."
1,"[(Robin Williams, 2157), (Jonathan Hyde, 8537)..."
2,"[(Walter Matthau, 6837), (Jack Lemmon, 3151), ..."
3,"[(Whitney Houston, 8851), (Angela Bassett, 978..."
4,"[(Steve Martin, 67773), (Diane Keaton, 3092), ..."


In [31]:
def get_directors_from_crew(credits):
    """
    Extracts the names of the director(s) from the 'crew' column of a Pandas DataFrame.

    Args:
        credits (pandas.DataFrame): A Pandas DataFrame containing 'crew' column with crew information.

    Returns:
        pandas.DataFrame: DataFrame with a column 'director_name' containing the names of the director(s), and the corresponding director 'id', and 'credits id' for the film.

    """
    # Set 'id' column as index
    global director_credit_id
    credits = credits.set_index('id')

    director_info = []

    for index, row in credits.iterrows():  # Iterate using index and row
        crew_list_str = row['crew']
        crew_list = ast.literal_eval(crew_list_str)
        director_name = None
        director_id = None
        film_id = index  # Get the film 'id' (now index)

        for crew_member_l in crew_list:
            if crew_member_l['job'] == 'Director':
                director_name = crew_member_l['name']
                director_credit_id = crew_member_l['credit_id']
                break

        director_info.append((director_name, director_credit_id, film_id))

    return pd.DataFrame({'director_info': director_info})


director_info_df = get_directors_from_crew(credits_df)
director_info_df.head()


,director_info
0,"(John Lasseter, 52fe4284c3a36847f8024f49, 862)"
1,"(Joe Johnston, 52fe44bfc3a36847f80a7c7d, 8844)"
2,"(Howard Deutch, 52fe466a9251416c75077a89, 15602)"
3,"(Forest Whitaker, 52fe44779251416c91011acb, 31..."
4,"(Charles Shyer, 52fe44959251416c75039eef, 11862)"


In [32]:
def get_first_3_crew(crew_series):
    """
    Extracts names and IDs of the first 3 crew members from a Pandas Series.

    Args:
        crew_series (pandas.Series): A Pandas Series containing crew information
                                      (string representation of list of dictionaries).

    Returns:
        pandas.DataFrame: DataFrame with columns 'crew_name' and 'crew_id' for the first 3 crew members.
    """

    all_crew_info = []

    for crew_list_str in crew_series:
        crew_list_s = ast.literal_eval(crew_list_str)  # Convert string to list
        crew_info = []
        for i in range(min(3, len(crew_list_s))):
            crew_info.append((crew_list_s[i]['name'], crew_list_s[i]['job'], crew_list_s[i]['id']))  # Create (name, job, ID) tuple
        all_crew_info.append(crew_info)

    return pd.DataFrame({'crew_info': all_crew_info})


crew_info_df = get_first_3_crew(credits_df['crew'])
# crew_info_df.head()

In [33]:
# find all the unique jobs of from crew_info_df
# dataframe example: [(John Lasseter, Director, 7879), (Joss Whedon...
unique_jobs = set()
for crew_list in crew_info_df['crew_info']:
    for crew_member in crew_list:
        unique_jobs.add(crew_member[1])

In [256]:
top_3_credits_df = pd.concat([cast_info_df, director_info_df, credits_df['id']], axis=1)
top_3_credits_df.head()

,cast_info,director_info,id
0,"[(Tom Hanks, 31), (Tim Allen, 12898), (Don Ric...","(John Lasseter, 52fe4284c3a36847f8024f49, 862)",862
1,"[(Robin Williams, 2157), (Jonathan Hyde, 8537)...","(Joe Johnston, 52fe44bfc3a36847f80a7c7d, 8844)",8844
2,"[(Walter Matthau, 6837), (Jack Lemmon, 3151), ...","(Howard Deutch, 52fe466a9251416c75077a89, 15602)",15602
3,"[(Whitney Houston, 8851), (Angela Bassett, 978...","(Forest Whitaker, 52fe44779251416c91011acb, 31...",31357
4,"[(Steve Martin, 67773), (Diane Keaton, 3092), ...","(Charles Shyer, 52fe44959251416c75039eef, 11862)",11862


In [145]:
top_3_credits_df.shape

(45476, 3)

In [35]:
# movies_df.head()
# Make a separate table for genres with the genre name and the corresponding id. Take it from the genres column on movies_df
# Create a separate table for genres with the genre name and the corresponding id
import json

pre_genre_steal_df = movies_df.copy()

def extract_genres():
    genres_set = set()
    for genres_str in movies_df['genres']:
        genres_list = json.loads(genres_str.replace("'", "\""))
        for genre in genres_list:
            genres_set.add((genre['id'], genre['name']))
    return pd.DataFrame(genres_set, columns=['genre_id', 'genre_name'])

genres_df = extract_genres()
drop_entries = [7759, 7760, 7761, 11602, 11176, 33751, 29812, 2883]
genres_df = genres_df[~genres_df['genre_id'].isin(drop_entries)]
genres_df

,genre_id,genre_name
0,10751,Family
1,27,Horror
2,18,Drama
3,16,Animation
4,53,Thriller
5,80,Crime
6,37,Western
7,9648,Mystery
10,36,History
14,10770,TV Movie


In [ ]:
movies_df 

In [ ]:
ohe_movies_df = movies_df.copy()
ohe_movies_df.head(3)

In [39]:
# Bad genres: (7759,GoHands), (7760,BROSTA TV), (7761,Mardock Scramble Production Committee), (11602,Vision View Entertainment), (11176,Carousel Productions), (33751,Sentai Filmworks), (29812,Telescene Film Group Productions)
drop_entries = [7759, 7760, 7761, 11602, 11176, 33751, 29812, 2883]
# Drop the genre entries with id's listed above
ohe_movies_df['genres'] = ohe_movies_df['genres'].apply(
    lambda x: json.loads(x.replace("'", "\"")) if isinstance(x, str) else x
).apply(
    lambda x: [genre for genre in x if genre['id'] not in drop_entries] if isinstance(x, list) else []
)

# List all the genres (validation check - there's no dodgy genres left in)
all_genres = set()
for genres in ohe_movies_df['genres']:
    for genre in genres:
        all_genres.add(genre['name'])
print(all_genres)

{'Drama', 'Romance', 'Western', 'History', 'TV Movie', 'Comedy', 'Adventure', 'Music', 'Science Fiction', 'Fantasy', 'Family', 'Mystery', 'War', 'Animation', 'Documentary', 'Thriller', 'Horror', 'Action', 'Foreign', 'Crime'}


In [40]:
# Extract genre names
ohe_movies_df['genre_list'] = ohe_movies_df['genres'].apply(lambda x: [genre['name'] for genre in x] if isinstance(x, list) else [])
ohe_movies_df.head()

,genres,id,popularity,release_date,title,vote_average,vote_count,genre_list
26559,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",76600,6.020055,2020-12-16,Avatar 2,0.0,58.0,"[Action, Adventure, Fantasy, Science Fiction]"
38885,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",299782,0.238154,2018-12-31,The Other Side of the Wind,0.0,1.0,"[Comedy, Drama]"
30402,"[{'id': 53, 'name': 'Thriller'}, {'id': 28, 'n...",38700,2.178546,2018-11-07,Bad Boys for Life,0.0,12.0,"[Thriller, Action, Crime]"
38130,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",332283,3.328261,2018-04-25,Mary Shelley,0.0,1.0,"[Drama, Romance]"
44535,"[{'id': 18, 'name': 'Drama'}]",412059,0.155147,2018-04-04,Mobile Homes,0.0,1.0,[Drama]


In [41]:
# One-hot encode genres
from sklearn.preprocessing import MultiLabelBinarizer

# Initialize MultiLabelBinarizer
mlb = MultiLabelBinarizer()

genre_ohe = pd.DataFrame(mlb.fit_transform(ohe_movies_df['genre_list']),
                         columns=mlb.classes_,
                         index=ohe_movies_df.index)

# Merge back into sample_genres_movies_df
ohe_movies_df = pd.concat([ohe_movies_df, genre_ohe], axis=1)

genre_list_mlb = mlb.classes_
genre_list = ohe_movies_df['genre_list']

# Drop unnecessary columns
ohe_movies_df = ohe_movies_df.drop(columns=['genres', 'genre_list'])

# print(sample_genres_movies_df.head())
ohe_movies_df

,id,popularity,release_date,title,vote_average,vote_count,Action,Adventure,Animation,Comedy,...,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
26559,76600,6.020055,2020-12-16,Avatar 2,0.0,58.0,1,1,0,0,...,0,0,0,0,0,1,0,0,0,0
38885,299782,0.238154,2018-12-31,The Other Side of the Wind,0.0,1.0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
30402,38700,2.178546,2018-11-07,Bad Boys for Life,0.0,12.0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
38130,332283,3.328261,2018-04-25,Mary Shelley,0.0,1.0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
44535,412059,0.155147,2018-04-04,Mobile Homes,0.0,1.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45148,438910,0.001586,NaN,Engineering Red,6.0,2.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
45203,433711,0.00022,NaN,All Superheroes Must Die 2: The Last Superhero,4.0,1.0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
45338,335251,0.0,NaN,The Land Where the Blues Began,0.0,0.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
45410,449131,0.008903,NaN,Aprel,6.0,1.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [44]:
ohe_movies_df.sort_values(by='id')

,id,popularity,release_date,title,vote_average,vote_count,Action,Adventure,Animation,Comedy,...,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
2429,100,4.60786,1998-03-05,"Lock, Stock and Two Smoking Barrels",7.5,1671.0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
13609,10000,0.281609,1993-12-25,La estrategia del caracol,7.2,9.0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4435,10001,2.562888,1988-12-15,Young Einstein,4.5,46.0,0,0,0,1,...,0,0,0,0,0,1,0,0,0,0
17451,100010,0.769266,1940-12-27,Flight Command,6.0,1.0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
36946,100017,2.964103,2006-08-06,Hounded,4.8,7.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25652,99946,0.202315,1926-11-06,Exit Smiling,8.5,2.0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3767,9995,1.316179,2000-09-06,Turn It Up,5.0,5.0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12549,9997,3.840024,2007-11-15,Gabriel,5.0,77.0,1,0,0,0,...,0,1,0,0,0,1,0,0,0,0
25079,99977,0.215778,1979-08-10,Hot Stuff,7.8,6.0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [45]:
# ratings_df.shape

In [46]:
# userId_ratings = ratings_df.groupby('userId').size().reset_index(name='count')
# userId_ratings = userId_ratings.sort_values(by='userId', ascending=False)
# userId_ratings.head()


# Creating a user profile

In [245]:
user_ratings = """
710,golden eye, 4
1770,Michael Collins, 5
76600,Avatar 2, 3
141052,Justice League, 2.5
284053,Thor: Ragnarok, 4
341013,Atomic Blonde, 3.8
374720,Dunkirk, 4.7
339403,Baby Driver, 4.7
324852,Despicable Me 3, 4
419192,McLaren, 3.5
283995,Guardians of the Galaxy Vol. 2, 3.8
324849,The Lego Batman Movie, 4.5
324552,John Wick: Chapter 2, 3.5
180863,T2 Trainspotting, 3.6
318846,The Big Short, 4.4
406,La Haine, 4.65
424,Schindler's List, 4.8
627,Trainspotting, 4.9
107,Snatch, 4.9
1429,25th Hour, 3.3
49517,Tinker Tailor Soldier Spy, 4.1
6538,Charlie Wilson's War, 3.9
10315,Fantastic Mr. Fox, 4.7
27205,Inception, 3.6
106646,The Wolf of Wall Street, 4.5
120467,The Grand Budapest Hotel, 4.6
157336,Interstellar, 3.9
261023,Black Mass, 3.6
278,The Shawshank Redemption, 3
4232,Scream, 2.4
1893,Star Wars: Episode I - The Phantom Menace, 3.8
550,Fight Club, 4.2
98,Gladiator, 4
508,Love Actually, 2.1
228967,The Interview, 2.7
1895,Star Wars: Episode III - Revenge of the Sith, 3.5
18785,The Hangover, 1.9
67913,The Guard, 3.7
40807,50/50, 2.5
70160,The Hunger Games, 2.4
77930,Magic Mike, 1
72190,World War Z, 2.5
187017,22 Jump Street, 1.9
198663,The Maze Runner, 2
207703,Kingsman: The Secret Service, 2.1
99861,Avengers: Age of Ultron, 2
167073,Brooklyn, 2.8
254470,Pitch Perfect 2, 1.9
271718,Trainwreck, 1
314365,Spotlight, 4.9
259693,The Conjuring 2, 2.3
308266,War Dogs, 2.1
324786,Hacksaw Ridge, 3.1
330459,Rogue One: A Star Wars Story, 2.9
339846,Baywatch, 2
"""

len(user_ratings)

# 100,"Lock, Stock and Two Smoking Barrels", 4.3

1465

In [221]:
from io import StringIO


# Convert into a DataFrame
user_ratings_df = pd.read_csv(StringIO(user_ratings), header=None, names=["movieId", "movie_name", "rating"])

# Drop the movie_name column as it's not needed for appending to the ratings DataFrame
user_ratings_df = user_ratings_df.drop(columns=["movie_name"])

user_ratings_df['movieId'] = pd.to_numeric(user_ratings_df['movieId'], errors='coerce')
user_ratings_df = user_ratings_df.dropna(subset=['movieId'])
user_ratings_df['movieId'] = user_ratings_df['movieId'].astype(int)

# Testing user id = 999,999
user_ratings_df["userId"] = 999999

# Reorder columns to match the existing ratings DataFrame
user_ratings_df = user_ratings_df[["userId", "movieId", "rating"]]


# Print the DataFrame
print(user_ratings_df.head(3))


   userId  movieId  rating
0  999999      710     4.0
1  999999     1770     5.0
2  999999    76600     3.0


In [246]:
len(user_ratings_df)

55

In [222]:
ratings_df = pd.concat([ratings_df, user_ratings_df], ignore_index=True)
# ratings_df = ratings_df.drop('timestamp', axis=1) 
ratings_df.head()
# ratings_df.tail(10)


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077.0
1,1,25,1.0,944250228.0
2,1,29,2.0,943230976.0
3,1,30,5.0,944249077.0
4,1,32,5.0,943228858.0


In [242]:
# print film 107 info


KeyError: 'id'

In [260]:



def create_user_profile(user_id, ratings_df, top_3_credits_df):
  """Creates a user profile based on ratings for movies with shared cast/directors."""
  directors = []
  user_ratings = ratings_df[ratings_df['userId'] == user_id]
  # print(user_ratings)
  profile = {}

  for _, rating_row in user_ratings.iterrows():
    movie_id = rating_row['movieId']
    rating = rating_row['rating']

    #find the index of the movie_id in top_3_credits_df
    try:
        movie_index = top_3_credits_df[top_3_credits_df['id'] == movie_id].index[0]
    except IndexError:
        print(f"Movie ID {movie_id} not found in top_3_credits_df")
        continue # if the movie_id isn't in top_3_credits_df, skip this movie


    # Add cast and director IDs to user profile with weighted ratings
    cast_ids = top_3_credits_df['cast_info'].iloc[movie_index]
    for cast_member in cast_ids:
        profile[cast_member[1]] = profile.get(cast_member[1], 0) + rating

    director_info = top_3_credits_df['director_info'].iloc[movie_index]
    if director_info is not None:
        profile[director_info[0]] = profile.get(director_info[0], 0) + rating
        directors.append(director_info[0])
    
    # Add genre IDs to user profile with weighted ratings
    genre_score = 0.1 * rating
    for genre in mlb.classes_:
        if ohe_movies_df[genre].iloc[movie_index] == 1:
            profile[genre] = profile.get(genre, 0) + genre_score
    

  return profile, directors


# Example usage:
test_user_id = 999999  # Replace it with your desired user ID
user_profile, directors_list = create_user_profile(test_user_id, ratings_df, top_3_credits_df)
user_profile
# len(directors_list)

{517: 4.0,
 48: 4.0,
 10695: 4.0,
 'Martin Campbell': 4.0,
 'Comedy': 5.655000000000001,
 3896: 13.600000000000001,
 18992: 5.0,
 9029: 5.0,
 'Neil Jordan': 5.0,
 'Horror': 2.43,
 'Mystery': 2.7,
 'Thriller': 3.950000000000001,
 204: 3.0,
 8691: 6.8,
 65731: 6.1,
 'James Cameron': 3.0,
 'Crime': 1.78,
 'Romance': 2.9000000000000004,
 880: 2.5,
 73968: 2.5,
 90633: 2.5,
 'Zack Snyder': 2.5,
 'Drama': 7.440000000000002,
 74568: 6.0,
 91606: 4.0,
 112: 4.0,
 'Taika Waititi': 4.0,
 6885: 3.8,
 5530: 3.8,
 568657: 3.8,
 'David Leitch': 3.8,
 'Music': 0.5900000000000001,
 1687041: 4.7,
 1765227: 4.7,
 1334638: 4.7,
 'Christopher Nolan': 12.200000000000001,
 1159982: 4.7,
 1016168: 4.7,
 1979: 4.7,
 'Edgar Wright': 4.7,
 'Adventure': 2.1,
 'War': 0.78,
 4495: 8.4,
 41091: 4.0,
 34517: 4.0,
 'Kyle Balda': 4.0,
 'Western': 0.7100000000000001,
 218565: 3.5,
 90424: 3.5,
 1209543: 3.5,
 'Roger Donaldson': 3.5,
 'Documentary': 0.63,
 73457: 3.8,
 543530: 3.8,
 'James Gunn': 3.8,
 'Action': 2.94,
 

### User User Collaborative Filtering

In [261]:
ratings_cpy = ratings_df.copy()


In [262]:
# # Sort ratings_df by timestamp (most recent) and then translate it into a data format
# ratings_cpy = ratings_cpy.sort_values(by='timestamp', ascending=False)
# ratings_cpy['date'] = pd.to_datetime(ratings_cpy['timestamp'], unit='s')
# ratings_cpy.head()

In [263]:
ohe_movies_df[ohe_movies_df['id'] == 107]

,id,popularity,release_date,title,vote_average,vote_count,Action,Adventure,Animation,Comedy,...,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
3886,107,14.014025,2000-09-01,Snatch,7.7,2953.0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [264]:
from lenskit.algorithms import Recommender
from lenskit.algorithms.user_knn import UserUser

In [265]:
ratings_cpy = ratings_cpy.drop('timestamp', axis=1)

In [228]:


num_recs = 100  #<---- This is the number of recommendations to generate. You can change this if you want to see more recommendations



ratings_cpy.rename(columns={'userId': 'user'}, inplace=True)
ratings_cpy.rename(columns={'movieId': 'item'}, inplace=True)


user_user = UserUser(20, min_nbrs=2) #These two numbers set the minimum (3) and maximum (15) number of neighbours to consider. These are considered "reasonable defaults", but you can experiment with others too
algo = Recommender.adapt(user_user)
algo.fit(ratings_cpy)

print("Set up a User-User algorithm!")

Set up a User-User algorithm!


In [266]:
def get_user_user_recs(user_id, num_ids = 400):
    return algo.recommend(user_id, num_ids)

In [267]:
cathal_recs = algo.recommend(999999, num_recs)  #Here, -1 tells it that it's not an existing user in the set, that we're giving new ratings, while 10 is how many recommendations it should generate

# Print recommended movies
# print(cathal_recs)
cathal_recs
# ohe_movies_df[ohe_movies_df['id'].isin(cathal_recs)][['title', 'release_date', 'vote_average', 'vote_count']]

,item,score
0,159123,6.437773
1,157146,6.004664
2,175763,5.994524
3,172975,5.849234
4,150012,5.840783
...,...,...
95,104119,5.008313
96,201502,5.007318
97,119440,5.001450
98,176729,5.000862


In [268]:
ohe_movies_df[ohe_movies_df['id'].isin(cathal_recs.item)][['title', 'release_date', 'vote_average', 'vote_count']]

,title,release_date,vote_average,vote_count
34833,La mossa del pinguino,2014-03-01,5.3,25.0
26127,Grave Halloween,2013-10-19,4.4,28.0
41347,Love 911,2012-12-19,6.8,14.0
40422,Rat Fever,2012-06-22,0.0,0.0
33070,The Saint of Gamblers,1995-06-28,3.5,2.0
38618,The Gypsy and the Gentleman,1958-01-15,0.0,0.0
32146,K.O. Miguel,1957-01-01,0.0,0.0
19433,We Live Again,1934-11-01,0.0,0.0


In [269]:
# save_ohe_movies_df = ohe_movies_df.copy()

In [270]:
print("Number of matching movie IDs:", len(set(ohe_movies_df['id']) & set(top_3_credits_df['id'])))

# Convert the 'id' columns to numeric, forcing errors to NaN
ohe_movies_df.loc[:, 'id'] = pd.to_numeric(ohe_movies_df['id'], errors='coerce')
top_3_credits_df['id'] = pd.to_numeric(top_3_credits_df['id'], errors='coerce')

# Drop rows with NaN values in the 'id' columns
ohe_movies_df = ohe_movies_df.dropna(subset=['id'])
top_3_credits_df = top_3_credits_df.dropna(subset=['id'])

# Convert the 'id' columns to integers
ohe_movies_df.loc[:, 'id'] = ohe_movies_df['id'].astype(int)
top_3_credits_df['id'] = top_3_credits_df['id'].astype(int)

# Find common IDs
common_ids = set(ohe_movies_df['id']) & set(top_3_credits_df['id'])

print("Number of matching movie IDs:", len(common_ids))


Number of matching movie IDs: 45432
Number of matching movie IDs: 45432


# Generate recommendations


In [271]:
def score_breakdown(films_df, recommended_movies):
    return films_df[films_df['id'].isin([movie_id for movie_id, _, _, _, _, _ in recommended_movies])][['title', 'release_date', 'vote_average', 'vote_count']].assign(
    score=[score for _, score, _, _, _, _ in recommended_movies], 
    cast_score=[cast_score for _, _, cast_score, _, _, _ in recommended_movies], 
    director_score=[director_score for _, _, _, director_score, _, _ in recommended_movies], 
    genre_score=[genre_score for _, _, _, _, genre_score, _ in recommended_movies],
    user_user_score=[user_user_score for _, _, _, _, _, user_user_score in recommended_movies]
    )
    

### Content-Based Filtering

In [278]:
# I want to see how many movies actually have a director that matches with the user profile directors
# Use directors_list
# I want to see how many movies actually have a director that matches with the user profile directors
matching_directors_count = top_3_credits_df[top_3_credits_df['director_info'].apply(lambda x: x[0] if x is not None else None).isin(directors_list)].shape[0]
print(f"Number of movies with matching directors: {matching_directors_count}")

Number of movies with matching directors: 478


In [286]:
cast_ft_weight = 0.3
director_ft_weight = 0.3
genre_ft_weight = 0.15
user_user_weight = 1



def recommend_movies(user_id, user_profile, films, top_3_credits_df, directors, ratings_df, top_n=20):
    """Recommends movies based on user profile and movie cast/director."""

    user_rated_movies = set(ratings_df[ratings_df['userId'] == user_id]['movieId'])
    unrated_movies = films[~films['id'].isin(user_rated_movies)]
        
    print("Unrated movies dimensions:", unrated_movies.shape)
    recommendations = []
    
    user_user_scores = get_user_user_recs(user_id, 900)
    print("User-User scores retrieved!")

    for _, movie_row in unrated_movies.iterrows():
        movie_id = movie_row['id']
        score, cast_score, director_score, genre_score, user_user_score = 0, 0, 0, 0, 0
        
        
        # Append on user-user collaborative filtering score
        if movie_id in user_user_scores.item.values:
            user_user_score = user_user_scores[user_user_scores['item'] == movie_id].score.values[0]

        score += user_user_score * user_user_weight
        
        
        #find the index of the movie_id in top_3_credits_df
        try:
            movie_index = top_3_credits_df[top_3_credits_df['id'] == movie_id].index[0]
        except IndexError:
            print(f"Movie ID {movie_id} not found in top_3_credits_df")
            continue
        
                
        # Calculate weighted score based on user profile and movie cast/director, genres, 
        cast_ids = top_3_credits_df['cast_info'].iloc[movie_index]
        for cast_member in cast_ids:
            cast_score += user_profile.get(cast_member[1], 0) * cast_ft_weight
        score += cast_score
        
        director_info = directors.iloc[movie_index]
        # if director_info is not None:
        director_score += user_profile.get(director_info[0], 0) * director_ft_weight
        # if director_info[0] in directors_list:
        #     print(f"Movie ID: {movie_id}, Director Score: {director_score}")
        # if director_score > 0:
        #     print(f"Movie ID: {movie_id}, Director Score: {director_score}")
        score += director_score
        
        # Calculate weighted score based on one hot-encoded genres
        # TODO - need to make this a normalised score - e.g. if you have 7 genres that all, you're greatly dominating other films with 1 genre that matches
        genre_score = sum([user_profile.get(genre, 0) for genre in mlb.classes_ if movie_row[genre] == 1])
        score += genre_score * genre_ft_weight
        
        
               
        # print(f"Movie ID: {movie_id}, Score: {score}")
        recommendations.append((movie_id, score, cast_score, director_score, genre_score, user_user_score))
    
    recommendations.sort(key=lambda x: x[1], reverse=True)  # Sort by score
    # print(recommendations)
    top_recommendations = recommendations[:top_n]
    
    movie_recs = [(movie_id, score, cast_score, director_score, genre_score, user_user_score) for movie_id, score, cast_score, director_score, genre_score, user_user_score in top_recommendations]
    # least_rated_movie_ids = [movie_id for movie_id, score in recommendations[:top_n]]
    
    # recommendations.sort(key=lambda x: x[1])  # Sort in ascending order
    lowest_recommendations = recommendations[-top_n:]
    least_movie_recs = [(movie_id, score, cast_score, director_score, genre_score, user_user_score) for movie_id, score, cast_score, director_score, genre_score, user_user_score in lowest_recommendations]
    
    return movie_recs, least_movie_recs

In [287]:
recommended_movies, least_rated_movie_ids = recommend_movies(test_user_id, user_profile, ohe_movies_df, top_3_credits_df, top_3_credits_df['director_info'], ratings_df, top_n=50)


Unrated movies dimensions: (45408, 26)
User-User scores retrieved!
Movie ID 401840 not found in top_3_credits_df


In [288]:
score_breakdown(ohe_movies_df, recommended_movies)

,title,release_date,vote_average,vote_count,score,cast_score,director_score,genre_score,user_user_score
36242,Miles Ahead,2016-03-20,6.7,74.0,11.128500,7.980,2.19,6.390,0.000000
35696,Jane Got a Gun,2016-01-01,5.4,293.0,10.884000,5.400,3.66,12.160,0.000000
28624,Run All Night,2015-03-11,6.3,1169.0,10.608750,4.740,2.55,22.125,0.000000
27477,Mortdecai,2015-01-21,5.4,1078.0,9.673500,8.010,0.00,11.090,0.000000
25857,Son of a Gun,2014-10-16,6.1,284.0,9.610500,8.730,0.00,5.870,0.000000
24640,Foxcatcher,2014-05-19,6.5,965.0,9.403500,5.940,1.35,14.090,0.000000
20829,This Is the End,2013-06-12,6.2,2394.0,9.045000,8.730,0.00,2.100,0.000000
20721,Trance,2013-03-27,6.5,975.0,9.043500,7.290,0.00,11.690,0.000000
41347,Love 911,2012-12-19,6.8,14.0,8.923500,6.060,0.75,14.090,0.000000
18252,The Dark Knight Rises,2012-07-16,7.6,9263.0,8.626500,2.550,3.66,16.110,0.000000


In [ ]:
directors = top_3_credits_df['director_info']
directors[directors.apply(lambda x: x[1] if x is not None else None).isin(directors_list)]

In [113]:
# Sort recommended films by director score
recommended_movies.sort(key=lambda x: x[3], reverse=True)
# 
ohe_movies_df[ohe_movies_df['id'].isin([movie_id for movie_id, _, _, _, _, _ in recommended_movies])][['title', 'release_date', 'vote_average', 'vote_count']].assign(
    score=[score for _, score, _, _, _, _ in recommended_movies], 
    cast_score=[cast_score for _, _, cast_score, _, _, _ in recommended_movies], 
    director_score=[director_score for _, _, _, director_score, _, _ in recommended_movies], 
    genre_score=[genre_score for _, _, _, _, genre_score, _ in recommended_movies],
    user_user_score=[user_user_score for _, _, _, _, _, user_user_score in recommended_movies]
)


,title,release_date,vote_average,vote_count,score,cast_score,director_score,genre_score,user_user_score
38753,Sausage Party,2016-07-11,5.6,2310.0,9.673500,8.01,0.0,11.090,0.000000
36242,Miles Ahead,2016-03-20,6.7,74.0,9.610500,8.73,0.0,5.870,0.000000
35696,Jane Got a Gun,2016-01-01,5.4,293.0,9.045000,8.73,0.0,2.100,0.000000
28624,Run All Night,2015-03-11,6.3,1169.0,9.043500,7.29,0.0,11.690,0.000000
27477,Mortdecai,2015-01-21,5.4,1078.0,8.938500,7.98,0.0,6.390,0.000000
25857,Son of a Gun,2014-10-16,6.1,284.0,8.173500,6.06,0.0,14.090,0.000000
24640,Foxcatcher,2014-05-19,6.5,965.0,8.058750,4.74,0.0,22.125,0.000000
26127,Grave Halloween,2013-10-19,4.4,28.0,8.053500,5.94,0.0,14.090,0.000000
21713,Crystal Fairy & the Magical Cactus,2013-07-12,5.8,65.0,7.836000,6.72,0.0,7.440,0.000000
41347,Love 911,2012-12-19,6.8,14.0,7.817974,2.07,0.0,9.385,4.340224


In [66]:
# Jane Got a Gun
# Son of a Gun
# Foxcatcher
# Wrath of the Titans
# Moneyball
# Perfect Sense
# Beginners
# Clash of the Titans
# Shutter Island
# The Men Who Stare at Goats



# Sausage Party
# Miles Ahead
# Jane Got a Gun
# Mortdecai
# Foxcatcher
# Wrath of the Titans
# Moneyball
# Perfect Sense
# Clash of the Titans
# Shutter Island
# The Ghost Writer
# The Men Who Stare at Goats
# Angels & Demons
# Seraphim Falls
# Stay
# Batman Begins
# Star Wars: Episode II - Attack of the Clones
# Velvet Goldmine
# Before and After
# Rob Roy


In [67]:
ohe_movies_df[ohe_movies_df['id'].isin(least_rated_movie_ids)][['title', 'release_date', 'vote_average', 'vote_count']]


,title,release_date,vote_average,vote_count


Next, let's look at the ratings dataframe.


In [68]:
#Drop removes a specified row or column from a dataframe
ratings_df = ratings_df.drop('timestamp', axis=1)
ratings_df.head()

,userId,movieId,rating
0,1,110,1.0
1,1,147,4.5
2,1,858,5.0
3,1,1221,5.0
4,1,1246,5.0
